# 1) Load packages

In [49]:
import pandas as pd
import numpy as np

# 2) Load and clean data

In [6]:
#loading required columns only (from flights.csv) with selected data types to save memory.

cols = ["YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE", "FLIGHT_NUMBER",
        "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "SCHEDULED_DEPARTURE",
        "DEPARTURE_DELAY", "ARRIVAL_DELAY", "DISTANCE", "CANCELLED",
        "DIVERTED", "CANCELLATION_REASON", "AIR_SYSTEM_DELAY", "SECURITY_DELAY",
        "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]

dtypes = {
    "AIRLINE": "category",
    "ORIGIN_AIRPORT": "category",     
    "DESTINATION_AIRPORT": "category",
    "CANCELLATION_REASON": "category",
    "CANCELLED": "int8",
    "DIVERTED": "int8",
    "DAY_OF_WEEK": "int8",
    "MONTH": "int8",
    "DAY": "int8",
}

flights = pd.read_csv('flights.csv', usecols = lambda c: c in cols, dtype = dtypes)

In [5]:
#make copy of flights df as parquet for speed
flights.to_parquet("flights.parquet", index=False)

In [50]:
#########THIS LINE FOR ONCE CSV FLIGHTS HAS BEEN INITALLY LOADED IN, AND CONVERTED TO PARQUET
flights_df = pd.read_parquet("flights.parquet")

In [51]:
#reading in the two smaller datasets
airlines = pd.read_csv('airlines.csv')
airports = pd.read_csv('airports.csv')

In [52]:
#sanity checks
flights_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 20 columns):
 #   Column               Dtype   
---  ------               -----   
 0   YEAR                 int64   
 1   MONTH                int8    
 2   DAY                  int8    
 3   DAY_OF_WEEK          int8    
 4   AIRLINE              category
 5   FLIGHT_NUMBER        int64   
 6   ORIGIN_AIRPORT       category
 7   DESTINATION_AIRPORT  category
 8   SCHEDULED_DEPARTURE  int64   
 9   DEPARTURE_DELAY      float64 
 10  DISTANCE             int64   
 11  ARRIVAL_DELAY        float64 
 12  DIVERTED             int8    
 13  CANCELLED            int8    
 14  CANCELLATION_REASON  category
 15  AIR_SYSTEM_DELAY     float64 
 16  SECURITY_DELAY       float64 
 17  AIRLINE_DELAY        float64 
 18  LATE_AIRCRAFT_DELAY  float64 
 19  WEATHER_DELAY        float64 
dtypes: category(4), float64(7), int64(4), int8(5)
memory usage: 549.5 MB


In [53]:
#getting % of null values
flights_df.isna().mean().sort_values(ascending=False).head(10)

CANCELLATION_REASON    0.984554
AIRLINE_DELAY          0.817250
SECURITY_DELAY         0.817250
WEATHER_DELAY          0.817250
AIR_SYSTEM_DELAY       0.817250
LATE_AIRCRAFT_DELAY    0.817250
ARRIVAL_DELAY          0.018056
DEPARTURE_DELAY        0.014805
DAY_OF_WEEK            0.000000
DAY                    0.000000
dtype: float64

In [54]:
#getting raw counts for na/null values
flights_df.isna().sum()

YEAR                         0
MONTH                        0
DAY                          0
DAY_OF_WEEK                  0
AIRLINE                      0
FLIGHT_NUMBER                0
ORIGIN_AIRPORT               0
DESTINATION_AIRPORT          0
SCHEDULED_DEPARTURE          0
DEPARTURE_DELAY          86153
DISTANCE                     0
ARRIVAL_DELAY           105071
DIVERTED                     0
CANCELLED                    0
CANCELLATION_REASON    5729195
AIR_SYSTEM_DELAY       4755640
SECURITY_DELAY         4755640
AIRLINE_DELAY          4755640
LATE_AIRCRAFT_DELAY    4755640
WEATHER_DELAY          4755640
dtype: int64

In [61]:
flights_df.head(5)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,...,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,DATE,DEPARTURE_HOUR
0,2015,1,1,4,AS,98,ANC,SEA,5,-11.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0
1,2015,1,1,4,AA,2336,LAX,PBI,10,-8.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0
2,2015,1,1,4,US,840,SFO,CLT,20,-2.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0
3,2015,1,1,4,AA,258,LAX,MIA,20,-5.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0
4,2015,1,1,4,AS,135,SEA,ANC,25,-1.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0


In [56]:
# creating a date column
flights_df['DATE'] = pd.to_datetime(flights_df[['YEAR', 'MONTH', 'DAY']])

In [57]:
flights_df['SCHEDULED_DEPARTURE'].min()

1

In [58]:
flights_df['SCHEDULED_DEPARTURE'].max()

2359

In [59]:
# creating a depature time that is in hours as scheduled depature uses HHHH format
flights_df['DEPARTURE_HOUR'] = flights_df['SCHEDULED_DEPARTURE'] // 100

In [60]:
airlines.head(5)

,IATA_CODE,AIRLINE
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.
3,F9,Frontier Airlines Inc.
4,B6,JetBlue Airways


In [62]:
airports.head(5)

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447


# 3) Merge datasets

In [63]:
# renaming columns before the merge for easier to understand column names
airlines = airlines.rename(columns={
    'IATA_CODE': 'AIRLINE',
    'AIRLINE': 'AIRLINE_NAME'
})

In [64]:
flights_updated = flights_df.merge(airlines, on='AIRLINE', how='left')

In [65]:
#sanity check, number of rows have not changed
assert len(flights_updated) == len(flights_df), 'Number of rows changed after merge'

In [66]:
print(flights_updated['AIRLINE_NAME'].isna().sum())

0


In [67]:
flights_updated.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,...,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,DATE,DEPARTURE_HOUR,AIRLINE_NAME
0,2015,1,1,4,AS,98,ANC,SEA,5,-11.0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0,Alaska Airlines Inc.
1,2015,1,1,4,AA,2336,LAX,PBI,10,-8.0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0,American Airlines Inc.
2,2015,1,1,4,US,840,SFO,CLT,20,-2.0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0,US Airways Inc.
3,2015,1,1,4,AA,258,LAX,MIA,20,-5.0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0,American Airlines Inc.
4,2015,1,1,4,AS,135,SEA,ANC,25,-1.0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0,Alaska Airlines Inc.


In [68]:
# merging airports columns to get origin and destination airport names, cities, and states
airports_keep = airports[['IATA_CODE', 'AIRPORT', 'CITY', 'STATE']]

In [69]:
#creating an origin airports df
airports_origin = airports_keep.rename(columns = {
    'IATA_CODE': 'ORIGIN_AIRPORT',
    'AIRPORT': 'ORIGIN_AIRPORT_NAME',
    'CITY': 'ORIGIN_CITY',
    'STATE': 'ORIGIN_STATE'
})

In [70]:
#creating a destinations airports df
airports_destination = airports_keep.rename(columns = {
    'IATA_CODE': 'DESTINATION_AIRPORT',
    'AIRPORT': 'DESTINATION_AIRPORT_NAME',
    'CITY': 'DESTINATION_CITY',
    'STATE': 'DESTINATION_STATE'
})

In [71]:
# merging the origin and destination airport names, cities, and states to the flights_updated df to get the final merged df
flights_full = flights_updated.merge(airports_origin, on = 'ORIGIN_AIRPORT', how = 'left')

In [72]:
flights_full = flights_full.merge(airports_destination, on = 'DESTINATION_AIRPORT', how = 'left')

In [73]:
flights_full.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,...,WEATHER_DELAY,DATE,DEPARTURE_HOUR,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,DESTINATION_AIRPORT_NAME,DESTINATION_CITY,DESTINATION_STATE
0,2015,1,1,4,AS,98,ANC,SEA,5,-11.0,...,NaN,2015-01-01,0,Alaska Airlines Inc.,Ted Stevens Anchorage International Airport,Anchorage,AK,Seattle-Tacoma International Airport,Seattle,WA
1,2015,1,1,4,AA,2336,LAX,PBI,10,-8.0,...,NaN,2015-01-01,0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,Palm Beach International Airport,West Palm Beach,FL
2,2015,1,1,4,US,840,SFO,CLT,20,-2.0,...,NaN,2015-01-01,0,US Airways Inc.,San Francisco International Airport,San Francisco,CA,Charlotte Douglas International Airport,Charlotte,NC
3,2015,1,1,4,AA,258,LAX,MIA,20,-5.0,...,NaN,2015-01-01,0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,Miami International Airport,Miami,FL
4,2015,1,1,4,AS,135,SEA,ANC,25,-1.0,...,NaN,2015-01-01,0,Alaska Airlines Inc.,Seattle-Tacoma International Airport,Seattle,WA,Ted Stevens Anchorage International Airport,Anchorage,AK


In [74]:
#sanity check, number of rows have not changed
assert len(flights_full) == len(flights_df), 'Number of rows changed after merge'

# 4) Creating a consistent airport identifer

In [75]:
flights_full.isna().sum()

YEAR                              0
MONTH                             0
DAY                               0
DAY_OF_WEEK                       0
AIRLINE                           0
FLIGHT_NUMBER                     0
ORIGIN_AIRPORT                    0
DESTINATION_AIRPORT               0
SCHEDULED_DEPARTURE               0
DEPARTURE_DELAY               86153
DISTANCE                          0
ARRIVAL_DELAY                105071
DIVERTED                          0
CANCELLED                         0
CANCELLATION_REASON         5729195
AIR_SYSTEM_DELAY            4755640
SECURITY_DELAY              4755640
AIRLINE_DELAY               4755640
LATE_AIRCRAFT_DELAY         4755640
WEATHER_DELAY               4755640
DATE                              0
DEPARTURE_HOUR                    0
AIRLINE_NAME                      0
ORIGIN_AIRPORT_NAME          486165
ORIGIN_CITY                  486165
ORIGIN_STATE                 486165
DESTINATION_AIRPORT_NAME     486165
DESTINATION_CITY            

In [76]:
# lots of na/missing destination and origin airport names, so checking to see if airport codes switch/are not consistent throughout the dataset
flights_full['ORIGIN_AIRPORT'].nunique()

628

In [31]:
#628 unique airports yet the airports dataset only has 322 unique airports. this means at some point the flights dataset changes the formatting
#for origin and destination airports

In [77]:
print(flights_full['ORIGIN_AIRPORT'].unique())

['ANC' 'LAX' 'SFO' 'SEA' 'LAS' 'DEN' 'SLC' 'PDX' 'FAI' 'MSP' 'PHX' 'SJU'
 'PBG' 'IAG' 'PSE' 'BQN' 'ORD' 'GEG' 'HNL' 'ONT' 'MCO' 'BOS' 'HIB' 'ABR'
 'MAF' 'DFW' 'MKE' 'IAH' 'BNA' 'BRO' 'VPS' 'BOI' 'BJI' 'SGF' 'PHL' 'SBN'
 'RDD' 'EUG' 'IAD' 'BUF' 'PWM' 'JFK' 'CRP' 'PIA' 'FAT' 'SMF' 'AUS' 'MCI'
 'ATL' 'JAX' 'MFR' 'IDA' 'MSN' 'DCA' 'SAT' 'CHS' 'SBA' 'SMX' 'IND' 'CLE'
 'GSP' 'BDL' 'ABI' 'RIC' 'BFL' 'OMA' 'RDM' 'FLL' 'CID' 'TPA' 'SYR' 'ROC'
 'TYR' 'LAN' 'XNA' 'GSO' 'EWR' 'PBI' 'RSW' 'OAK' 'PVD' 'RNO' 'PIT' 'ABQ'
 'MIA' 'BWI' 'LGA' 'TUL' 'LIT' 'MSY' 'OKC' 'ATW' 'PNS' 'MEM' 'TYS' 'MHT'
 'SAV' 'CLT' 'GRB' 'ABE' 'JAN' 'OAJ' 'FAR' 'ERI' 'LEX' 'CWA' 'MSO' 'TTN'
 'AMA' 'CLL' 'HOU' 'JLN' 'MLI' 'RDU' 'CVG' 'MHK' 'MOB' 'TLH' 'BHM' 'CAE'
 'TXK' 'ACY' 'DTW' 'RAP' 'TUS' 'EAU' 'DLH' 'FSD' 'INL' 'CMX' 'SPI' 'CLD'
 'COD' 'CMH' 'LRD' 'PSC' 'CPR' 'ACV' 'DAL' 'PAH' 'MRY' 'ESC' 'ISN' 'PSP'
 'MFE' 'STL' 'BTV' 'FSM' 'AEX' 'SPS' 'ACT' 'SJT' 'MTJ' 'GCC' 'OGG' 'SJC'
 'GUC' 'ORF' 'MOT' 'MLU' 'KOA' 'SAN' 'LAW' 'PIB' 'M

In [78]:
#reading in the master coord supply table, time start converting the numeric airport didgit ids to IATA codes, to get a full dataset of airport names
master_data = pd.read_csv('AV_Supp_T_MASTER_CORD_20211207.csv')

In [79]:
#getting the latest id for each airport
lookup = master_data[master_data['AIRPORT_IS_LATEST'] == 1]

In [80]:
# creating a dictionary, each key is airport ID, and each value is that airports IATA code. The IDs in my dataset are being registered as strings so
# but in the csv there are digits, so astype(str) ensures consistent data types when converting to IATA codes. Zip is used to pair sequences element by
# element
id_to_IATA = dict(zip(lookup['AIRPORT_ID'].astype(str), lookup['AIRPORT']))

In [81]:
# map the ids to their corresponding airport IATA codes, and use fillna to keep the existing IATA codes that are not in ID form
for col in ['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']:
    flights_full[col] = flights_full[col].map(id_to_IATA).fillna(flights_full[col])

In [82]:
flights_full['ORIGIN_AIRPORT'].unique()

array(['ANC', 'LAX', 'SFO', 'SEA', 'LAS', 'DEN', 'SLC', 'PDX', 'FAI',
       'MSP', 'PHX', 'SJU', 'PBG', 'IAG', 'PSE', 'BQN', 'ORD', 'GEG',
       'HNL', 'ONT', 'MCO', 'BOS', 'HIB', 'ABR', 'MAF', 'DFW', 'MKE',
       'IAH', 'BNA', 'BRO', 'VPS', 'BOI', 'BJI', 'SGF', 'PHL', 'SBN',
       'RDD', 'EUG', 'IAD', 'BUF', 'PWM', 'JFK', 'CRP', 'PIA', 'FAT',
       'SMF', 'AUS', 'MCI', 'ATL', 'JAX', 'MFR', 'IDA', 'MSN', 'DCA',
       'SAT', 'CHS', 'SBA', 'SMX', 'IND', 'CLE', 'GSP', 'BDL', 'ABI',
       'RIC', 'BFL', 'OMA', 'RDM', 'FLL', 'CID', 'TPA', 'SYR', 'ROC',
       'TYR', 'LAN', 'XNA', 'GSO', 'EWR', 'PBI', 'RSW', 'OAK', 'PVD',
       'RNO', 'PIT', 'ABQ', 'MIA', 'BWI', 'LGA', 'TUL', 'LIT', 'MSY',
       'OKC', 'ATW', 'PNS', 'MEM', 'TYS', 'MHT', 'SAV', 'CLT', 'GRB',
       'ABE', 'JAN', 'OAJ', 'FAR', 'ERI', 'LEX', 'CWA', 'MSO', 'TTN',
       'AMA', 'CLL', 'HOU', 'JLN', 'MLI', 'RDU', 'CVG', 'MHK', 'MOB',
       'TLH', 'BHM', 'CAE', 'TXK', 'ACY', 'DTW', 'RAP', 'TUS', 'EAU',
       'DLH', 'FSD',

In [83]:
flights_full['DESTINATION_AIRPORT'].unique()

array(['SEA', 'PBI', 'CLT', 'MIA', 'ANC', 'MSP', 'DFW', 'ATL', 'IAH',
       'PDX', 'MCI', 'FLL', 'ORD', 'HNL', 'PHX', 'EWR', 'JFK', 'MCO',
       'BOS', 'BDL', 'ITO', 'SFO', 'KOA', 'OGG', 'MYR', 'DTW', 'LIH',
       'DEN', 'SJU', 'LAX', 'BWI', 'IAD', 'BQN', 'BUF', 'LGA', 'HOU',
       'SLC', 'PHL', 'SJC', 'OAK', 'LGB', 'TPA', 'DCA', 'TTN', 'BTR',
       'LAS', 'RSW', 'BRD', 'STL', 'RKS', 'MBS', 'SNA', 'MEI', 'MDW',
       'SAN', 'RIC', 'AUS', 'OTZ', 'PIT', 'JAX', 'MSY', 'ONT', 'PSP',
       'BUR', 'DAL', 'CVG', 'SMF', 'RDU', 'JMS', 'BNA', 'DSM', 'MAF',
       'BOI', 'ELP', 'TUS', 'SCC', 'HPN', 'STT', 'MDT', 'RHI', 'SBP',
       'MKE', 'JNU', 'CMH', 'CLD', 'KTN', 'CAK', 'CRP', 'CLE', 'GPT',
       'SHV', 'TYS', 'IND', 'LIT', 'SAT', 'SRQ', 'TUL', 'GRK', 'PNS',
       'BTV', 'CHS', 'DAY', 'OKC', 'SAV', 'XNA', 'COS', 'GJT', 'BZN',
       'PUB', 'HRL', 'HDN', 'MEM', 'GEG', 'ORH', 'SYR', 'GSO', 'VPS',
       'LAW', 'ACY', 'LBB', 'JAC', 'BIL', 'EUG', 'ASE', 'TVC', 'MTJ',
       'CAE', 'PVD',

In [84]:
flights_full.isna().sum()

YEAR                              0
MONTH                             0
DAY                               0
DAY_OF_WEEK                       0
AIRLINE                           0
FLIGHT_NUMBER                     0
ORIGIN_AIRPORT                    0
DESTINATION_AIRPORT               0
SCHEDULED_DEPARTURE               0
DEPARTURE_DELAY               86153
DISTANCE                          0
ARRIVAL_DELAY                105071
DIVERTED                          0
CANCELLED                         0
CANCELLATION_REASON         5729195
AIR_SYSTEM_DELAY            4755640
SECURITY_DELAY              4755640
AIRLINE_DELAY               4755640
LATE_AIRCRAFT_DELAY         4755640
WEATHER_DELAY               4755640
DATE                              0
DEPARTURE_HOUR                    0
AIRLINE_NAME                      0
ORIGIN_AIRPORT_NAME          486165
ORIGIN_CITY                  486165
ORIGIN_STATE                 486165
DESTINATION_AIRPORT_NAME     486165
DESTINATION_CITY            

# 5) remerging datasets as the airport identifier is now consistently in the IATA code form
### the merges will follow the same principles as chapter 3) but now with the correct data formatting

In [85]:
#dropping the columns that were merged before airport id became consistent (except origina and destination airport as these are now consistent)
old_cols = ['ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY', 'ORIGIN_STATE', 'DESTINATION_AIRPORT_NAME', 'DESTINATION_CITY', 'DESTINATION_STATE']
flights_full = flights_full.drop(columns=old_cols)

In [86]:
#remerging the datasets
flights_full = flights_full.merge(airports_origin, on = 'ORIGIN_AIRPORT', how = 'left')

In [87]:
flights_full = flights_full.merge(airports_destination, on = 'DESTINATION_AIRPORT', how = 'left')

In [88]:
flights_full.isna().sum()

YEAR                              0
MONTH                             0
DAY                               0
DAY_OF_WEEK                       0
AIRLINE                           0
FLIGHT_NUMBER                     0
ORIGIN_AIRPORT                    0
DESTINATION_AIRPORT               0
SCHEDULED_DEPARTURE               0
DEPARTURE_DELAY               86153
DISTANCE                          0
ARRIVAL_DELAY                105071
DIVERTED                          0
CANCELLED                         0
CANCELLATION_REASON         5729195
AIR_SYSTEM_DELAY            4755640
SECURITY_DELAY              4755640
AIRLINE_DELAY               4755640
LATE_AIRCRAFT_DELAY         4755640
WEATHER_DELAY               4755640
DATE                              0
DEPARTURE_HOUR                    0
AIRLINE_NAME                      0
ORIGIN_AIRPORT_NAME               0
ORIGIN_CITY                       0
ORIGIN_STATE                      0
DESTINATION_AIRPORT_NAME          0
DESTINATION_CITY            

In [89]:
flights_full.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,...,WEATHER_DELAY,DATE,DEPARTURE_HOUR,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,DESTINATION_AIRPORT_NAME,DESTINATION_CITY,DESTINATION_STATE
0,2015,1,1,4,AS,98,ANC,SEA,5,-11.0,...,NaN,2015-01-01,0,Alaska Airlines Inc.,Ted Stevens Anchorage International Airport,Anchorage,AK,Seattle-Tacoma International Airport,Seattle,WA
1,2015,1,1,4,AA,2336,LAX,PBI,10,-8.0,...,NaN,2015-01-01,0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,Palm Beach International Airport,West Palm Beach,FL
2,2015,1,1,4,US,840,SFO,CLT,20,-2.0,...,NaN,2015-01-01,0,US Airways Inc.,San Francisco International Airport,San Francisco,CA,Charlotte Douglas International Airport,Charlotte,NC
3,2015,1,1,4,AA,258,LAX,MIA,20,-5.0,...,NaN,2015-01-01,0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,Miami International Airport,Miami,FL
4,2015,1,1,4,AS,135,SEA,ANC,25,-1.0,...,NaN,2015-01-01,0,Alaska Airlines Inc.,Seattle-Tacoma International Airport,Seattle,WA,Ted Stevens Anchorage International Airport,Anchorage,AK
